# Qwen3-4B-FP8 smoke test (non-thinking)

Minimal check that `Qwen/Qwen3-4B-FP8` loads and generates on a single prompt with thinking mode disabled (`enable_thinking=False`), using the canonical Hugging Face usage.

The model is pinned to one GPU via `device_map="cuda:0"` — `device_map="auto"` offloads layers to CPU on a busy node and tanks decode to a few tok/s. The tok/s readout shows whether FP8 is actually fast here.

(First run downloads `Qwen/Qwen3-4B-FP8`, ~4 GB.)

In [ ]:
# Import tamart FIRST so it sets HF_HOME (-> data/hf) before transformers loads.
import tamart  # noqa: F401
import time

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

In [ ]:
model_name = "Qwen/Qwen3-4B"

t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(model_name)
# device_map="cuda:0" pins the whole model to one GPU. Do NOT use "auto" on a
# shared node: it offloads layers to CPU based on free memory and decode crawls.
# (FP8 weights need accelerate dispatch, so use device_map rather than .to().)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="cuda:0",
    attn_implementation="flash_attention_2",
)
print(f"loaded in {time.time() - t0:.1f}s | device: {model.device} | dtype: {model.dtype}")
print("attn impl:", model.config._attn_implementation)

In [ ]:
prompt = "Write 2000 tokens bro"
messages = [{"role": "user", "content": prompt}]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,  # non-thinking mode
    use_cache=True,
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

max_new_tokens = 512  # small cap for a quick smoke test

t0 = time.time()
generated_ids = model.generate(**model_inputs, max_new_tokens=max_new_tokens)
dt = time.time() - t0

output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()
content = tokenizer.decode(output_ids, skip_special_tokens=True)

n_new = len(output_ids)
print(f"generated {n_new} tokens in {dt:.1f}s -> {n_new / dt:.1f} tok/s\n")
print("content:", content)